In [0]:
%skip
from pyspark.sql import functions as F

In [0]:
BRONZE = "abfss://bronze@pharmadatalakeuk.dfs.core.windows.net"
SILVER = "abfss://silver@pharmadatalakeuk.dfs.core.windows.net"
mes = (spark.read
       .option("header", "true")
       .csv(f"{BRONZE}/mes/mes_batches.csv"))
mes_dedup = mes.dropDuplicates()
print(f"Rows: {mes.count()}")
print(f"After :{mes_dedup.count()}")
print(mes_dedup.select("batch_id").distinct().count())

In [0]:
#Trim for spaces in all columns using for loop
mes_trimmed = mes_dedup.select([F.trim(F.col(c)).alias(c) for c in mes_dedup.columns])

In [0]:
#mapping different values in column "deviation_flag" --> Yes or No
# If theere is no value that doesnot count it as No --> stays as null
deviation_flag_clean = (
    F.when(F.upper(F.col("deviation_flag")).isin("Y","YES","TRUE","1"), "Yes")
     .when(F.upper(F.col("deviation_flag")).isin("N","NO","FALSE","0"), "No")
)

mes_standardised = mes_trimmed.withColumn("site", F.initcap("site"))\
                .withColumn("production_line", F.upper("production_line"))\
                .withColumn("product_code", F.upper("product_code"))\
                .withColumn("batch_status", F.initcap(F.lower("batch_status")))\
                .withColumn("deviation_flag", deviation_flag_clean)
display(mes_standardised)

In [0]:
#Casting the columns with proper datatypes


#removing th unwanted % symbol using regex and then using coalesce to add avalues if yield_pct is null. --> this is a calculated filed that is used when yield_pct is null
yield_pct_clean = F.coalesce(
    F.regexp_replace(F.col("yield_pct"), "%", "").try_cast("double"),
    F.round(F.col("actual_qty").try_cast("double") / F.col("planned_qty").try_cast("double") * 100, 2)
)

mes_typed = mes_standardised.withColumn("planned_qty", F.col("planned_qty").try_cast("double"))\
                .withColumn("actual_qty", F.col("actual_qty").try_cast("double"))\
                .withColumn("yield_pct", yield_pct_clean)

# mes_typed.printSchema()

In [0]:
print("Yield null but actual present:",
      mes_typed.filter(F.col("yield_pct").isNull() & F.col("actual_qty").isNotNull()).count())

print("yield_pct nulls:", mes_typed.filter(F.col("yield_pct").isNull()).count())
print("actual_qty nulls:", mes_typed.filter(F.col("actual_qty").isNull()).count())

In [0]:
# display(
#     mes_typed.groupBy("batch_status")
#     .agg(
#         F.count("*").alias("rows"),
#         F.sum(F.col("actual_qty").isNull().cast("int")).alias("actual_null"),
#         F.sum(F.col("yield_pct").isNull().cast("int")).alias("yield_null"),
#     )
# )

In [0]:
def parse_timestamp(column_name):
    col = F.col(column_name)
    return F.coalesce(
        F.try_to_timestamp(col),
        F.try_to_timestamp(col, F.lit("dd/MM/yyyy HH:mm")),
        F.try_to_timestamp(col, F.lit("dd-MMM-yyyy HH:mm")),
    )

In [0]:
mes_parsed = mes_typed.withColumn("start_time", parse_timestamp("start_time"))\
                .withColumn("end_time", parse_timestamp("end_time"))
mes_parsed.printSchema()

In [0]:
F.expr("try_to_timestamp(start_time, 'yyyy-MM-dd HH:mm:ss')")

In [0]:
# print("Unparsed start_time:", mes_parsed.filter(F.col("start_time").isNull()).count())
# print("Unparsed end_time:  ", mes_parsed.filter(F.col("end_time").isNull()).count())

display(mes_parsed.select("batch_id", "batch_status", "start_time", "end_time").limit(10))

In [0]:
mes_final = (mes_parsed\
    .withColumn("batch_duration_hours",
                F.round((F.col("end_time").cast("long") - F.col("start_time").cast("long")) / 3600, 2))
    .withColumn("yield_exceeds_100", F.col("yield_pct") > 100)
    .withColumn("end_before_start", F.col("end_time") < F.col("start_time")))

In [0]:
print("Rows:", mes_final.count())
print("Yield above 100:", mes_final.filter(F.col("yield_exceeds_100")).count())
print("End before start:", mes_final.filter(F.col("end_before_start")).count())

mes_final.printSchema()

In [0]:
(mes_final.write
 .format("delta")
 .mode("overwrite")
 .save(f"{SILVER}/mes"))

# Plain Parquet copy for Snowflake. Overwrite removes old files, so only the current version is kept.
(mes_final.write
 .mode("overwrite")
 .parquet(f"{SILVER}/export/mes"))

check = spark.read.format("delta").load(f"{SILVER}/mes")
print("Written and read back:", check.count())

In [0]:
export_check = spark.read.parquet(f"{SILVER}/export/mes")
print("Export read back:", export_check.count())

In [0]:
# display(mes_final)